In [ ]:
# -*- coding: utf-8 -*-
"""
Bước 1: Tính toán khoảng cách Euclid và xác định láng giềng thủ công (chưa dùng thư viện).
Chạy trực tiếp trên VS Code (dùng đường dẫn tương đối).
"""

import numpy as np
import pandas as pd

# Khởi tạo dữ liệu mẫu cho 3 học viên
tập_họcvien = {
    "Học sinh A": np.array([5.0, 3.0, 4000.0]),
    "Học sinh B": np.array([25.0, 9.0, 4100.0]),
    "Học sinh C": np.array([6.0, 3.5, 7000.0]),
}

def tinh_khoang_cach_euclid(vec1, vec2):
    """Hàm tính khoảng cách Euclid giữa hai tập đặc trưng."""
    return np.sqrt(np.sum((vec1 - vec2) ** 2))

print("=== KHOẢNG CÁCH BAN ĐẦU (CHƯA CHUẨN HÓA) ===")
print(f" Khoảng cách (A, B) = {tinh_khoang_cach_euclid(tập_họcvien['Học sinh A'], tập_họcvien['Học sinh B']):8.1f}")
print(f" Khoảng cách (A, C) = {tinh_khoang_cach_euclid(tập_họcvien['Học sinh A'], tập_họcvien['Học sinh C']):8.1f}")
print()

# Đọc tập dữ liệu từ file CSV cùng thư mục làm việc trong VS Code
df_sv = pd.read_csv("sinh_vien_mo_rong.csv")
matran_X = df_sv[["gio_on", "diem_giua_ky", "chi_tieu"]].to_numpy(dtype=float)

trung_binh_cot = matran_X.mean(axis=0)
do_lech_chuan_cot = matran_X.std(axis=0)

print("--- Độ lệch chuẩn của từng trường dữ liệu ---")
print(f" Giờ ôn: {do_lech_chuan_cot[0]:.2f} | Điểm giữa kỳ: {do_lech_chuan_cot[1]:.2f} | Chi tiêu: {do_lech_chuan_cot[2]:.0f}")
print()

# Chuẩn hóa Z-score cho dữ liệu 3 học viên
chuan_hoa_họcvien = {ten: (gia_tri - trung_binh_cot) / do_lech_chuan_cot for ten, gia_tri in tập_họcvien.items()}

print("=== KHOẢNG CÁCH SAU KHI THỰC HIỆN CHUẨN HÓA ===")
print(f" Khoảng cách (A, B) = {tinh_khoang_cach_euclid(chuan_hoa_họcvien['Học sinh A'], chuan_hoa_họcvien['Học sinh B']):.2f}")
print(f" Khoảng cách (A, C) = {tinh_khoang_cach_euclid(chuan_hoa_họcvien['Học sinh A'], chuan_hoa_họcvien['Học sinh C']):.2f}")
print()

# Tìm kiếm 5 điểm dữ liệu gần nhất cho một sinh viên mới (chỉ xét 2 cột học tập)
X_hoctap = df_sv[["gio_on", "diem_giua_ky"]].to_numpy(dtype=float)
y_ketqua = df_sv["qua_mon"].to_numpy()

mean_ht, std_ht = X_hoctap.mean(axis=0), X_hoctap.std(axis=0)
thi_sinh_moi = np.array([12.0, 6.5])

X_hoctap_norm = (X_hoctap - mean_ht) / std_ht
thi_sinh_norm = (thi_sinh_moi - mean_ht) / std_ht

# Đếm khoảng cách từ thí sinh mới tới toàn bộ mẫu
danh_sach_kc = np.sqrt(((X_hoctap_norm - thi_sinh_norm) ** 2).sum(axis=1))
chi_so_k_gan_nhat = np.argsort(danh_sach_kc)[:5]

print("=== DỰ ĐOÁN CHO THÍ SINH MỚI (Giờ ôn: 12, Điểm GK: 6.5) ===")
print("5 láng giềng cận kề nhất:")
for idx in chi_so_k_gan_nhat:
    trang_thai = "ĐẠT" if y_ketqua[idx] == 1 else "TRƯỢT"
    print(f" - Số giờ: {X_hoctap[idx, 0]:5.1f} | Điểm GK: {X_hoctap[idx, 1]:4.1f} | Độ xa: {danh_sach_kc[idx]:.3f} -> Kết quả: {trang_thai}")

tong_dau = int(y_ketqua[chi_so_k_gan_nhat].sum())
print(f"Tỷ lệ bỏ phiếu: {tong_dau} ĐẠT, {5 - tong_dau} TRƯỢT")
print("=> Dự báo chung:", "ĐẠT MÔN" if tong_dau >= 3 else "TRƯỢT MÔN")


In [ ]:
# -*- coding: utf-8 -*-
"""
Bước 2: Đánh giá ảnh hưởng của chuẩn hóa dữ liệu tới hiệu năng mô hình trên tập test.
"""

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Đọc dữ liệu sinh viên
df_sv = pd.read_csv("sinh_vien_mo_rong.csv")
y_label = df_sv["qua_mon"]
X_features = df_sv[["gio_on", "diem_giua_ky", "chi_tieu"]]

# Phân chia tập huấn luyện và kiểm thử
X_train_set, X_test_set, y_train_set, y_test_set = train_test_split(
    X_features, y_label, test_size=0.25, random_state=17, stratify=y_label
)

def danh_gia_hieu_nang(mo_hinh_knn, danh_sach_cot):
    """Huấn luyện và tính độ chính xác cho mô hình."""
    mo_hinh_knn.fit(X_train_set[danh_sach_cot], y_train_set)
    du_bao = mo_hinh_knn.predict(X_test_set[danh_sach_cot])
    return accuracy_score(y_test_set, du_bao)

def khoi_tao_knn_standardized():
    """Tạo pipeline tích hợp chuẩn hóa Z-score và KNN."""
    return make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=7))

cot_day_du = ["gio_on", "diem_giua_ky", "chi_tieu"]
cot_rut_gon = ["gio_on", "diem_giua_ky"]

acc_chuachuanhoa = danh_gia_hieu_nang(KNeighborsClassifier(n_neighbors=7), cot_day_du)
acc_cochuanhoa = danh_gia_hieu_nang(khoi_tao_knn_standardized(), cot_day_du)
acc_bocolong = danh_gia_hieu_nang(khoi_tao_knn_standardized(), cot_rut_gon)

print("=== KẾT QUẢ ĐÁNH GIÁ ĐỘ CHÍNH XÁC (ACCURACY) ===")
print(f" - Dự đoán ngẫu nhiên theo đa số : {y_test_set.mean():.4f}")
print(f" - Đủ 3 cột (Không chuẩn hóa)      : {acc_chuachuanhoa:.4f}")
print(f" - Đủ 3 cột (Có chuẩn hóa)         : {acc_cochuanhoa:.4f}")
print(f" - Loại bỏ chi tiêu (Có chuẩn hóa) : {acc_bocolong:.4f}")


In [ ]:
# -*- coding: utf-8 -*-
"""
Bước 3: Lựa chọn tham số k tối ưu bằng kỹ thuật kiểm định chéo (Cross-Validation).
"""

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier

# Đọc dữ liệu bài toán hai vầng trăng
df_moon = pd.read_csv("hai_vang_trang.csv")
X_moon = df_moon[["x1", "x2"]]
y_moon = df_moon["nhan"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X_moon, y_moon, test_size=0.25, random_state=17, stratify=y_moon
)

print("=== BẢNG THỐNG KÊ ACCURACY THEO TỪNG THAM SỐ K ===")
print("   k      Tập học (Train)   Tập kiểm tra (Test)")
for k_val in [1, 5, 13, 31, 101]:
    clf = KNeighborsClassifier(n_neighbors=k_val).fit(X_tr, y_tr)
    acc_train = accuracy_score(y_tr, clf.predict(X_tr))
    acc_test = accuracy_score(y_te, clf.predict(X_te))
    print(f" {k_val:3d}        {acc_train:.4f}             {acc_test:.4f}")
print()

# Thực hiện 5-fold Cross-Validation trên tập Train để chọn k tốt nhất
print("=== KIỂM ĐỊNH CHÉO 5 PHẦN (5-FOLD CV) TRÊN TẬP TRAIN ===")
bang_diem_cv = {}
for k_val in range(1, 52, 2):
    cv_scores = cross_val_score(KNeighborsClassifier(n_neighbors=k_val), X_tr, y_tr, cv=5)
    bang_diem_cv[k_val] = cv_scores.mean()

k_toi_uu = max(bang_diem_cv, key=bang_diem_cv.get)

for k_val in [1, 5, 9, 13, 17, 25, 51]:
    danh_dau = " <= [TỐI ƯU NHẤT]" if k_val == k_toi_uu else ""
    print(f" k = {k_val:2d} | Điểm CV trung bình: {bang_diem_cv[k_val]:.4f}{danh_dau}")
print()

# Đánh giá lại k tối ưu trên tập kiểm tra (Test set)
mo_hinh_chinh = KNeighborsClassifier(n_neighbors=k_toi_uu).fit(X_tr, y_tr)
print(f"-> Tham số k chọn lựa: {k_toi_uu}")
do_chinh_xac_cuoi = accuracy_score(y_te, mo_hinh_chinh.predict(X_te))
print(f"-> Accuracy thực tế trên tập Test: {do_chinh_xac_cuoi:.4f}")


In [ ]:
# -*- coding: utf-8 -*-
"""
Bước 4: So sánh KNN với các thuật toán phân loại khác (Logistic Regression, Naive Bayes).
"""

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

df_moon = pd.read_csv("hai_vang_trang.csv")
X_data = df_moon[["x1", "x2"]]
y_data = df_moon["nhan"]

X_train, X_test, y_train, y_test = train_test_split(
    X_data, y_data, test_size=0.25, random_state=17, stratify=y_data
)

danh_sach_mo_hinh = [
    ("Hồi quy Logistic ", LogisticRegression()),
    ("Gaussian Naive Bayes", GaussianNB()),
    ("KNN (k = 13)       ", KNeighborsClassifier(n_neighbors=13)),
]

print("=== SO SÁNH HIỆU NĂNG CÁC MÔ HÌNH TRÊN DỮ LIỆU HAI VẦNG TRĂNG ===")
for ten_mh, algo in danh_sach_mo_hinh:
    algo.fit(X_train, y_train)
    acc_score = accuracy_score(y_test, algo.predict(X_test))
    print(f" + {ten_mh}: {acc_score:.4f}")


In [ ]:
# -*- coding: utf-8 -*-
"""
Bước 5: Áp dụng KNN cho bài toán Hồi quy (KNeighborsRegressor) trên bộ dữ liệu giá nhà.
"""

import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor

df_nhadat = pd.read_csv("gia_nha.csv")
X_nha = df_nhadat[["dien_tich"]]
y_nha = df_nhadat["gia"]

# Phân chia tập dữ liệu train/test với test_size=0.2
X_tr, X_te, y_tr, y_te = train_test_split(X_nha, y_nha, test_size=0.2, random_state=42)

reg_knn = KNeighborsRegressor(n_neighbors=5).fit(X_tr, y_tr)
reg_linear = LinearRegression().fit(X_tr, y_tr)

print("=== SAI SỐ TUYỆT ĐỐI TRUNG BÌNH (MAE) TRÊN TẬP TEST (TỶ ĐỒNG) ===")
mae_knn_val = mean_absolute_error(y_te, reg_knn.predict(X_te))
mae_linear_val = mean_absolute_error(y_te, reg_linear.predict(X_te))
print(f" - Mô hình KNN (k = 5)  : {mae_knn_val:.4f}")
print(f" - Mô hình Hồi quy tuyến tính: {mae_linear_val:.4f}")
print()

df_can_moi = pd.DataFrame({"dien_tich": [60.0, 100.0, 150.0, 250.0]})
pred_knn = reg_knn.predict(df_can_moi)
pred_linear = reg_linear.predict(df_can_moi)

print("=== DỰ BÁO GIÁ CHO 4 CĂN HỘ MỚI (TỶ ĐỒNG) ===")
print(" Diện tích     Dự báo KNN     Hồi quy tuyến tính")
for dt, p_k, p_l in zip(df_can_moi["dien_tich"], pred_knn, pred_linear):
    print(f" {dt:6.0f} m2      {p_k:8.2f}          {p_l:8.2f}")
print()

# Xác định 5 láng giềng cận kề nhất của căn nhà có diện tích 250 m2
_, indices = reg_knn.kneighbors(pd.DataFrame({"dien_tich": [250.0]}))
lang_gieng_list = X_tr.iloc[indices[0]]["dien_tich"].tolist()
print("Danh sách diện tích 5 láng giềng gần nhất với căn 250 m2:")
print(", ".join(f"{val:.1f} m2" for val in lang_gieng_list))
